# Notebook 12d — Heterogeneous vs Uniform Head Structure (MQAR Capacity)

## The Core Architectural Claim
This notebook tests the defining claim of the VLA v2 architecture:
> "Hetero heads maintain >0.8 accuracy at n_pairs=200 while Uniform heads collapse."

### Architecture (`d_model = 512`)

**1. Uniform (Baseline):** 4 heads × $d_h = 128$. Capacity threshold ≈ 128.

**2. Hetero (The Innovation):** 4 fast heads ($d_h = 16$) + 2 slow heads ($d_h = 224$).
Total $d_{model} = 4×16 + 2×224 = 64 + 448 = 512$. Capacity threshold ≈ 224.

**3. Gated-Hetero (Ablation):** Same as Hetero + learned forget gate for overload recovery.

### Attention Engine: DeltaNet Delta Rule
We use the DeltaNet delta rule ($S_t = S_{t-1} + \\beta (v_t - S_{t-1} k_t) k_t^\\top$) as the
attention engine because it is **proven** for associative recall tasks.
The VLA Sherman-Morrison recurrence is designed for language modeling;
the delta rule lets us cleanly isolate the **heterogeneous head structure** as the variable under test.

### Key Design Decisions
- **Read-before-write:** The recurrence reads $y_t = S q_t$ BEFORE writing,
  so query positions retrieve the correct association before any potential corruption.
- **Labels at key positions:** The loss is computed at the query-key position (where
  the model sees the key as input), not at the subsequent noise position.
  This is a discriminative setup, not autoregressive next-token prediction.
- **Adaptive sequence length:** Shorter sequences for smaller `n_pairs` to speed up training.


## 0 · Setup & Configuration

In [ ]:
import math, time, gc, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as torch_checkpoint

# Clear any zombie tensors held by Jupyter exceptions
if hasattr(sys, 'last_traceback'):
    sys.last_traceback = None
    sys.last_type = None
    sys.last_value = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT    = Path('/kaggle/working/nb12d_hetero_vla')
for sub in ['plots', 'logs']:
    (OUT/sub).mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'font.family': 'DejaVu Serif', 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25, 'figure.dpi': 150,
})

print(f'Device : {DEVICE}')

# ── Configuration ─────────────────────────────────────────────────────────
VOCAB   = 1024   # Large vocab to support up to 256 distinct keys/values
D_MODEL = 512
BATCH   = 64
N_LAYERS = 2     # 2 layers needed: L1 forms cross-position associations, L2 retrieves
CHUNK   = 32

# Sweep targets
N_PAIRS = [32, 64, 128, 160, 200, 256]
SEEDS   = [42, 123, 999]

LR      = 3e-4
WARMUP  = 300
GRAD_CLIP = 1.0

def seq_len_for_n(n):
    """Adaptive sequence length: shorter for fewer pairs → faster training."""
    half = max(n * 2, 256)   # Need n even slots (n*2 positions) in first half
    return max(half * 2, 512) # T = 2*half, min 512

def steps_for_n(n):
    """Step budget: more for harder tasks."""
    if n <= 32:  return 1500
    if n <= 64:  return 2000
    if n <= 128: return 3000
    if n <= 160: return 4000
    return 5000

print(f'VOCAB={VOCAB}  D_MODEL={D_MODEL}  BATCH={BATCH}  N_LAYERS={N_LAYERS}')
for n in N_PAIRS:
    print(f'  n={n:<4d} -> seq_len={seq_len_for_n(n):<5d} steps={steps_for_n(n):>5,d}')

## 1 · MQAR Generator

**Critical fix:** Labels are placed at the **query-key position** (where the model sees the key as input),
not at a subsequent noise position. This ensures the model has all the information it needs to predict the value.

In [ ]:
def make_mqar(B, n_pairs, vocab=VOCAB, seq_len=None, device=DEVICE):
    """Generate MQAR data with labels at query-key positions.
    
    Format:
      First half:  [k1, v1, k2, v2, ..., noise, ..., SEP]
      Second half: [query_k1, query_k2, ..., noise, ...]
      Labels:      [-100, ..., val_1, val_2, ..., -100, ...]
                   (labels only at query-key positions in second half)
    """
    if seq_len is None:
        seq_len = seq_len_for_n(n_pairs)
    
    sep       = vocab - 1
    noise_tok = 0
    key_lo, key_hi = 1, vocab // 2          # [1, 512): 511 distinct keys
    val_lo, val_hi = vocab // 2, vocab - 1  # [512, 1023): 511 distinct values
    
    n_keys = key_hi - key_lo
    assert n_pairs <= n_keys, f'n_pairs={n_pairs} > available keys={n_keys}'
    
    T = seq_len
    half = T // 2
    n_slots = half // 2  # Available even positions for KV pairs in first half
    assert n_pairs <= n_slots, f'n_pairs={n_pairs} > available slots={n_slots}'
    
    x = torch.full((B, T), noise_tok, dtype=torch.long, device=device)
    y = torch.full((B, T), -100, dtype=torch.long, device=device)
    
    for b in range(B):
        keys = torch.randperm(n_keys, device=device)[:n_pairs] + key_lo
        vals = torch.randint(val_lo, val_hi, (n_pairs,), device=device)
        
        # KV pairs in first half at random even positions
        positions = torch.randperm(n_slots, device=device)[:n_pairs] * 2
        for i in range(n_pairs):
            pos = positions[i].item()
            x[b, pos]     = keys[i]      # Key
            x[b, pos + 1] = vals[i]      # Value
        
        x[b, half - 1] = sep  # Separator
        
        # Queries in second half (contiguous, one per position)
        perm = torch.randperm(n_pairs, device=device)
        for i in range(n_pairs):
            qpos = half + i
            if qpos < T:
                x[b, qpos] = keys[perm[i]]   # Query key as input
                y[b, qpos] = vals[perm[i]]    # Expected value as label (AT KEY POSITION)
    
    return x, y

# Verify the generator
x_test, y_test = make_mqar(2, 32, seq_len=seq_len_for_n(32))
print(f'Test: x={x_test.shape}, labels at {(y_test[0] != -100).sum().item()} positions')
print(f'Expected ~random baseline loss: ln({VOCAB}) = {math.log(VOCAB):.2f}')

## 2 · Architectures

All three models share the same **DeltaNet delta rule** recurrence.
The ONLY variable is the **head structure** (uniform vs heterogeneous).

In [ ]:
def _split_raw(x, H, dh):
    B, T, D = x.shape
    return x.view(B, T, H, dh).permute(0, 2, 1, 3)

def _merge(x):
    B, H, T, dh = x.shape
    return x.permute(0, 2, 1, 3).reshape(B, T, H * dh)

# ══════════════════════════════════════════════════════════════════════════
# Delta Rule Recurrence (DeltaNet-style, proven for MQAR)
# ══════════════════════════════════════════════════════════════════════════
def _delta_recurrence(K, Q, V, beta, G, S, use_gate):
    """DeltaNet delta rule with READ-BEFORE-WRITE ordering.
    
    K: (B, H, Tc, dh) - L2-normalized key features
    Q: (B, H, Tc, dh) - query features (elu+1)
    V: (B, H, Tc, dh) - value vectors
    beta: (B, H, Tc, 1) - learned write strength in (0, 1)
    G: (B, H, Tc, 1) or None - forget gate
    S: (B, H, dh, dh) - state matrix (associative memory)
    """
    Tc = K.shape[2]
    ys = []
    for tc in range(Tc):
        k_t = K[:, :, tc, :]       # (B, H, dh)
        q_t = Q[:, :, tc, :]       # (B, H, dh)
        v_t = V[:, :, tc, :]       # (B, H, dh)
        b_t = beta[:, :, tc, :]    # (B, H, 1)
        
        # ── READ first (before any write at this step) ──
        y_t = torch.einsum('bhde,bhe->bhd', S, q_t)
        ys.append(y_t)
        
        # ── WRITE (error-corrective delta rule) ──
        e_t = v_t - torch.einsum('bhde,bhe->bhd', S, k_t)  # prediction error
        update = b_t.unsqueeze(-1) * torch.einsum('bhd,bhe->bhde', e_t, k_t)
        
        if use_gate:
            g_t = G[:, :, tc, :]   # (B, H, 1)
            S = g_t.unsqueeze(-1) * S + update
        else:
            S = S + update
    
    return torch.stack(ys, dim=2), S


# ══════════════════════════════════════════════════════════════════════════
# 1. UNIFORM (Baseline): 4 heads × d_h=128
# ══════════════════════════════════════════════════════════════════════════
class UniformDelta(nn.Module):
    def __init__(self, d_model=D_MODEL, H=4, use_gate=False):
        super().__init__()
        self.H = H; self.dh = d_model // H; self.use_gate = use_gate
        dh = self.dh
        
        self.Wq = nn.Parameter(torch.empty(H, dh, dh))
        self.Wk = nn.Parameter(torch.empty(H, dh, dh))
        self.Wv = nn.Parameter(torch.empty(H, dh, dh))
        self.bq = nn.Parameter(torch.zeros(H, dh))
        self.bk = nn.Parameter(torch.zeros(H, dh))
        self.bv = nn.Parameter(torch.zeros(H, dh))
        
        # Beta: learned scalar write strength per head per step
        self.Wb = nn.Parameter(torch.empty(H, dh, 1))
        self.bb = nn.Parameter(torch.zeros(H, 1))
        
        if use_gate:
            self.Wg = nn.Parameter(torch.empty(H, dh, 1))
            self.bg = nn.Parameter(torch.full((H, 1), 2.0))  # init ~sigmoid(2)=0.88
            nn.init.kaiming_uniform_(self.Wg, a=math.sqrt(5))
        
        for w in [self.Wq, self.Wk, self.Wv, self.Wb]:
            nn.init.kaiming_uniform_(w, a=math.sqrt(5))
        
        self.Wo = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)
    
    def forward(self, x):
        B, T, _ = x.shape; H, dh = self.H, self.dh
        xh = _split_raw(x, H, dh)
        
        Q = F.elu(torch.einsum('bhtd,hde->bhte', xh, self.Wq) + self.bq[None,:,None,:]) + 1.0
        K = F.elu(torch.einsum('bhtd,hde->bhte', xh, self.Wk) + self.bk[None,:,None,:]) + 1.0
        K = F.normalize(K, p=2, dim=-1)  # L2 norm for stable writes
        V = torch.einsum('bhtd,hde->bhte', xh, self.Wv) + self.bv[None,:,None,:]
        
        beta = torch.sigmoid(torch.einsum('bhtd,hde->bhte', xh, self.Wb) + self.bb[None,:,None,:])
        G = None
        if self.use_gate:
            G = torch.sigmoid(torch.einsum('bhtd,hde->bhte', xh, self.Wg) + self.bg[None,:,None,:])
        
        S = torch.zeros(B, H, dh, dh, device=x.device, dtype=x.dtype)
        
        ys = []
        for st in range(0, T, CHUNK):
            en = min(st + CHUNK, T)
            Gc = G[:,:,st:en] if G is not None else None
            out_c, S = torch_checkpoint.checkpoint(
                _delta_recurrence,
                K[:,:,st:en], Q[:,:,st:en], V[:,:,st:en], beta[:,:,st:en], Gc,
                S, self.use_gate, use_reentrant=False)
            ys.append(out_c)
        
        return self.Wo(self.norm(_merge(torch.cat(ys, dim=2))))


# ══════════════════════════════════════════════════════════════════════════
# 2. HETERO (The Innovation): 4 fast (d_h=16) + 2 slow (d_h=224)
# ══════════════════════════════════════════════════════════════════════════
class HeteroDelta(nn.Module):
    def __init__(self, d_model=D_MODEL, H1=4, dh1=16, H2=2, dh2=224, use_gate=False):
        super().__init__()
        self.H1, self.dh1 = H1, dh1
        self.H2, self.dh2 = H2, dh2
        self.D1 = H1 * dh1  # 64
        self.D2 = H2 * dh2  # 448
        self.use_gate = use_gate
        assert self.D1 + self.D2 == d_model, f'{self.D1}+{self.D2} != {d_model}'
        
        # Group 1: Fast heads
        self.Wq1 = nn.Parameter(torch.empty(H1, dh1, dh1))
        self.Wk1 = nn.Parameter(torch.empty(H1, dh1, dh1))
        self.Wv1 = nn.Parameter(torch.empty(H1, dh1, dh1))
        self.bq1 = nn.Parameter(torch.zeros(H1, dh1))
        self.bk1 = nn.Parameter(torch.zeros(H1, dh1))
        self.bv1 = nn.Parameter(torch.zeros(H1, dh1))
        self.Wb1 = nn.Parameter(torch.empty(H1, dh1, 1))
        self.bb1 = nn.Parameter(torch.zeros(H1, 1))
        
        # Group 2: Slow heads
        self.Wq2 = nn.Parameter(torch.empty(H2, dh2, dh2))
        self.Wk2 = nn.Parameter(torch.empty(H2, dh2, dh2))
        self.Wv2 = nn.Parameter(torch.empty(H2, dh2, dh2))
        self.bq2 = nn.Parameter(torch.zeros(H2, dh2))
        self.bk2 = nn.Parameter(torch.zeros(H2, dh2))
        self.bv2 = nn.Parameter(torch.zeros(H2, dh2))
        self.Wb2 = nn.Parameter(torch.empty(H2, dh2, 1))
        self.bb2 = nn.Parameter(torch.zeros(H2, 1))
        
        if use_gate:
            self.Wg1 = nn.Parameter(torch.empty(H1, dh1, 1))
            self.bg1 = nn.Parameter(torch.full((H1, 1), 2.0))
            self.Wg2 = nn.Parameter(torch.empty(H2, dh2, 1))
            self.bg2 = nn.Parameter(torch.full((H2, 1), 2.0))
            nn.init.kaiming_uniform_(self.Wg1, a=math.sqrt(5))
            nn.init.kaiming_uniform_(self.Wg2, a=math.sqrt(5))
        
        for w in [self.Wq1, self.Wk1, self.Wv1, self.Wb1,
                   self.Wq2, self.Wk2, self.Wv2, self.Wb2]:
            nn.init.kaiming_uniform_(w, a=math.sqrt(5))
        
        self.Wo = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)
    
    def _proj(self, xh, Wq, bq, Wk, bk, Wv, bv, Wb, bb, Wg, bg):
        H, dh = xh.shape[1], xh.shape[3]
        Q = F.elu(torch.einsum('bhtd,hde->bhte', xh, Wq) + bq[None,:,None,:]) + 1.0
        K = F.elu(torch.einsum('bhtd,hde->bhte', xh, Wk) + bk[None,:,None,:]) + 1.0
        K = F.normalize(K, p=2, dim=-1)
        V = torch.einsum('bhtd,hde->bhte', xh, Wv) + bv[None,:,None,:]
        beta = torch.sigmoid(torch.einsum('bhtd,hde->bhte', xh, Wb) + bb[None,:,None,:])
        G = None
        if self.use_gate and Wg is not None:
            G = torch.sigmoid(torch.einsum('bhtd,hde->bhte', xh, Wg) + bg[None,:,None,:])
        return K, Q, V, beta, G
    
    def forward(self, x):
        B, T, _ = x.shape
        x1, x2 = torch.split(x, [self.D1, self.D2], dim=-1)
        x1h = _split_raw(x1, self.H1, self.dh1)
        x2h = _split_raw(x2, self.H2, self.dh2)
        
        K1, Q1, V1, b1, G1 = self._proj(
            x1h, self.Wq1, self.bq1, self.Wk1, self.bk1, self.Wv1, self.bv1,
            self.Wb1, self.bb1, getattr(self,'Wg1',None), getattr(self,'bg1',None))
        K2, Q2, V2, b2, G2 = self._proj(
            x2h, self.Wq2, self.bq2, self.Wk2, self.bk2, self.Wv2, self.bv2,
            self.Wb2, self.bb2, getattr(self,'Wg2',None), getattr(self,'bg2',None))
        
        S1 = torch.zeros(B, self.H1, self.dh1, self.dh1, device=x.device, dtype=x.dtype)
        S2 = torch.zeros(B, self.H2, self.dh2, self.dh2, device=x.device, dtype=x.dtype)
        
        ys1, ys2 = [], []
        for st in range(0, T, CHUNK):
            en = min(st + CHUNK, T)
            G1c = G1[:,:,st:en] if G1 is not None else None
            G2c = G2[:,:,st:en] if G2 is not None else None
            
            o1, S1 = torch_checkpoint.checkpoint(
                _delta_recurrence,
                K1[:,:,st:en], Q1[:,:,st:en], V1[:,:,st:en], b1[:,:,st:en], G1c,
                S1, self.use_gate, use_reentrant=False)
            ys1.append(o1)
            
            o2, S2 = torch_checkpoint.checkpoint(
                _delta_recurrence,
                K2[:,:,st:en], Q2[:,:,st:en], V2[:,:,st:en], b2[:,:,st:en], G2c,
                S2, self.use_gate, use_reentrant=False)
            ys2.append(o2)
        
        out1 = _merge(torch.cat(ys1, dim=2))  # (B, T, D1)
        out2 = _merge(torch.cat(ys2, dim=2))  # (B, T, D2)
        return self.Wo(self.norm(torch.cat([out1, out2], dim=-1)))


# ── Backbone ──
class Block(nn.Module):
    def __init__(self, attn, d):
        super().__init__()
        self.ln1 = nn.LayerNorm(d); self.ln2 = nn.LayerNorm(d)
        self.attn = attn
        self.ff = nn.Sequential(nn.Linear(d, d*2), nn.GELU(), nn.Linear(d*2, d))
    def forward(self, x): return x + self.ff(self.ln2(x + self.attn(self.ln1(x))))

class TinyLM(nn.Module):
    def __init__(self, attn_fn, d=D_MODEL, vocab=VOCAB, n_layers=N_LAYERS):
        super().__init__()
        self.tok = nn.Embedding(vocab, d)
        nn.init.normal_(self.tok.weight, std=0.02)  # GPT-2 init prevents logit explosion
        self.pos = nn.Embedding(2048, d)
        nn.init.normal_(self.pos.weight, std=0.02)
        self.blocks = nn.ModuleList([Block(attn_fn(), d) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d)
        self.head = nn.Linear(d, vocab, bias=False)
        self.head.weight = self.tok.weight  # Weight tying
    def forward(self, idx):
        x = self.tok(idx) + self.pos(torch.arange(idx.shape[1], device=idx.device).unsqueeze(0))
        for b in self.blocks: x = b(x)
        return self.head(self.ln_f(x))

MODEL_REGISTRY = {
    'Uniform':      (lambda: UniformDelta(use_gate=False), '#74B9FF', 'D'),
    'Hetero':       (lambda: HeteroDelta(use_gate=False),  '#E17055', 'o'),
    'Gated-Hetero': (lambda: HeteroDelta(use_gate=True),   '#55EFC4', 's'),
}

for name, (fn, _, _) in MODEL_REGISTRY.items():
    m = TinyLM(fn)
    print(f'  {name}: {sum(p.numel() for p in m.parameters())/1e6:.2f}M params')
    del m

## 3 · Training Loop & Smoke Test

In [ ]:
def run_mqar(factory, n_pairs, steps, seed=42, log_every=100):
    # Safety GC at start of every run
    if hasattr(sys, 'last_traceback'):
        sys.last_traceback = None
        sys.last_type = None
        sys.last_value = None
    gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    
    seq_len = seq_len_for_n(n_pairs)
    
    torch.manual_seed(seed)
    model = TinyLM(factory).to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    
    def lrf(s):
        if s < WARMUP: return s / max(WARMUP, 1)
        return 0.5 * (1 + math.cos(math.pi*(s-WARMUP)/max(steps-WARMUP,1)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lrf)
    
    t0 = time.time()
    model.train()
    for step in range(1, steps + 1):
        opt.zero_grad(set_to_none=True)
        x, y = make_mqar(BATCH, n_pairs, seq_len=seq_len)
        loss = F.cross_entropy(model(x).view(-1, VOCAB), y.view(-1), ignore_index=-100)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        opt.step(); sched.step()
        
        if step % log_every == 0 or step == 1:
            model.eval()
            with torch.no_grad():
                xp, yp = make_mqar(32, n_pairs, seq_len=seq_len)
                mp = yp != -100
                probe = (model(xp).argmax(-1)[mp]==yp[mp]).float().mean().item() if mp.any() else 0.0
            model.train()
            print(f'      step {step:4d}/{steps}  loss={loss.item():.4f}  acc={probe:.4f}  ({time.time()-t0:.0f}s)')
    
    # Final Eval (10 batches for stable estimate)
    model.eval(); accs = []
    with torch.no_grad():
        for _ in range(10):
            xv, yv = make_mqar(32, n_pairs, seq_len=seq_len)
            mask = yv != -100
            if mask.any():
                accs.append((model(xv).argmax(-1)[mask] == yv[mask]).float().mean().item())
    
    final_acc = float(np.mean(accs))
    del model, opt; gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    return final_acc

print('--- SMOKE TEST: Hetero on n=32 (should reach >0.8 in 500 steps) ---')
acc = run_mqar(MODEL_REGISTRY['Hetero'][0], n_pairs=32, steps=500, log_every=50)
print(f'\nFinal accuracy: {acc:.4f}')

## 4 · Capacity Sweep (The Core Claim)

In [ ]:
results = []

for n in N_PAIRS:
    st = steps_for_n(n)
    print(f'\n{"="*60}\n n_pairs = {n}  (budget: {st} steps, seq_len={seq_len_for_n(n)})\n{"="*60}')
    for mname, (factory, col, mk) in MODEL_REGISTRY.items():
        print(f'\n  [{mname}]')
        accs = []
        for seed in SEEDS:
            acc = run_mqar(factory, n, st, seed=seed, log_every=st)
            accs.append(acc)
            print(f'    seed={seed}: {acc:.4f}')
        mean, std = np.mean(accs), np.std(accs)
        print(f'    -> MEAN = {mean:.4f} +/- {std:.4f}')
        results.append({'n_pairs': n, 'model': mname, 'mean': mean, 'std': std})

df = pd.DataFrame(results)
df.to_csv(OUT/'logs'/'hetero_capacity.csv', index=False)
print('\nResults saved to', OUT/'logs'/'hetero_capacity.csv')
df

## 5 · Plotting the Result

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for mname, (_, col, mk) in MODEL_REGISTRY.items():
    g = df[df.model == mname]
    if g.empty: continue
    ax.plot(g.n_pairs, g['mean'], color=col, marker=mk, lw=2.5, ms=9, label=mname)
    ax.fill_between(g.n_pairs, g['mean'] - g['std'], g['mean'] + g['std'], color=col, alpha=0.15)

# Capacity reference lines
ax.axvline(128, color='#74B9FF', ls='--', lw=1.5, alpha=0.7, label='Uniform max d_h (128)')
ax.axvline(224, color='#E17055', ls='--', lw=1.5, alpha=0.7, label='Hetero max d_h (224)')
ax.axhline(0.80, color='gray', ls=':', lw=1.5)
ax.text(250, 0.82, 'pass (0.80)', ha='right', fontsize=9, color='gray')

ax.set(title=f'MQAR Capacity: Hetero vs Uniform Heads\nd_model={D_MODEL}, delta rule engine',
       xlabel='n_pairs', ylabel='Accuracy')
ax.set_ylim(-0.02, 1.05)
ax.legend(fontsize=9)

plt.savefig(OUT/'plots'/'hetero_claim.png', bbox_inches='tight', dpi=200)
plt.show()
print('Plot saved to', OUT/'plots'/'hetero_claim.png')